In [2]:
import langchain
import langchain_community

print("LangChain:", langchain.__version__)
print("LangChain Community:", langchain_community.__version__)

LangChain: 1.3.17
LangChain Community: 0.4.2


In [3]:
import sys
print(sys.executable)

D:\arun\ai-learning\.venv\Scripts\python.exe


In [7]:
import os
import time
from pathlib import Path

from langchain_community.document_loaders import (
    TextLoader,
    PyPDFLoader
)

from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer

In [8]:
DOCUMENTS_DIR = Path("documents")

print(DOCUMENTS_DIR.resolve())

D:\arun\ai-learning\final_project\ingestion\documents


In [9]:
def load_documents(folder):
    documents = []

    for file_path in folder.iterdir():

        if file_path.suffix.lower() == ".txt":
            loader = TextLoader(
                str(file_path),
                encoding="utf-8"
            )
            docs = loader.load()

        elif file_path.suffix.lower() == ".pdf":
            loader = PyPDFLoader(str(file_path))
            docs = loader.load()

        else:
            print(f"Skipping unsupported file: {file_path.name}")
            continue

        for doc in docs:
            doc.metadata["source"] = file_path.name

        documents.extend(docs)

    return documents

In [10]:
documents = load_documents(DOCUMENTS_DIR)

print("Documents/pages loaded:", len(documents))

Documents/pages loaded: 13


In [11]:
print(documents[0].page_content[:1000])
print(documents[0].metadata)


      _____
     /     \
    | () () |
     \  ^  /
      |||||
      |||||

   /\_/\
  ( o.o )
   > ^ <

  __
 /  \
| () |
 \__/


This sample TXT file is provided by Sample-Files.com. Visit us for more sample files and resources.
{'source': 'ascii-art.txt'}


In [12]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150
)

In [13]:
chunks = text_splitter.split_documents(documents)

print("Original documents/pages:", len(documents))
print("Total chunks:", len(chunks))

Original documents/pages: 13
Total chunks: 147


In [14]:
for i, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = i

In [15]:
print(chunks[0].metadata)

{'source': 'ascii-art.txt', 'chunk_id': 0}


In [16]:
for i in [0, 1, 2]:
    print("=" * 80)
    print("CHUNK:", i)
    print("METADATA:", chunks[i].metadata)
    print("TEXT:")
    print(chunks[i].page_content[:500])

CHUNK: 0
METADATA: {'source': 'ascii-art.txt', 'chunk_id': 0}
TEXT:
_____
     /     \
    | () () |
     \  ^  /
      |||||
      |||||

   /\_/\
  ( o.o )
   > ^ <

  __
 /  \
| () |
 \__/


This sample TXT file is provided by Sample-Files.com. Visit us for more sample files and resources.
CHUNK: 1
METADATA: {'producer': 'Skia/PDF m126', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36', 'creationdate': '2024-07-09T13:31:41+00:00', 'title': 'Sample Document for PDF Testing', 'moddate': '2024-07-09T13:31:41+00:00', 'source': 'basic-text.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'chunk_id': 1}
TEXT:
Sample Document for PDF Testing
Introduction
This is a simple document created to test basic PDF functionality. It includes various text formatting
options to ensure proper rendering in PDF readers.
Text Formatting Examples
1. Bold text is used for emphasis.
2. Italic text can be used for titles or subtl

In [17]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [18]:
test_embedding = embedding_model.encode(
    "This is a test sentence."
)

print(test_embedding.shape)

(384,)


In [20]:
texts = [chunk.page_content for chunk in chunks]

start = time.perf_counter()

embeddings = embedding_model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True
)

embedding_time = time.perf_counter() - start

print("Embedding shape:", embeddings.shape)
print(f"Embedding time: {embedding_time:.2f} seconds")

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Embedding shape: (147, 384)
Embedding time: 12.20 seconds


In [21]:
import chromadb

client = chromadb.PersistentClient(
    path="chroma_db"
)

In [22]:
collection = client.get_or_create_collection(
    name="knowledge_base"
)

In [23]:
ids = [
    str(chunk.metadata["chunk_id"])
    for chunk in chunks
]

metadatas = [
    chunk.metadata
    for chunk in chunks
]

documents_text = [
    chunk.page_content
    for chunk in chunks
]

In [24]:
collection.add(
    ids=ids,
    documents=documents_text,
    embeddings=embeddings.tolist(),
    metadatas=metadatas
)

In [25]:
count = collection.count()

print("Stored chunks:", count)
print("Expected chunks:", len(chunks))

Stored chunks: 147
Expected chunks: 147


In [26]:
result = collection.get(
    limit=3,
    include=["documents", "metadatas"]
)

In [27]:
for i in range(len(result["ids"])):
    print("=" * 80)
    print("ID:", result["ids"][i])
    print("METADATA:", result["metadatas"][i])
    print("TEXT:", result["documents"][i][:500])

ID: 0
METADATA: {'source': 'ascii-art.txt', 'chunk_id': 0}
TEXT: _____
     /     \
    | () () |
     \  ^  /
      |||||
      |||||

   /\_/\
  ( o.o )
   > ^ <

  __
 /  \
| () |
 \__/


This sample TXT file is provided by Sample-Files.com. Visit us for more sample files and resources.
ID: 1
METADATA: {'creationdate': '2024-07-09T13:31:41+00:00', 'chunk_id': 1, 'page': 0, 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36', 'total_pages': 1, 'moddate': '2024-07-09T13:31:41+00:00', 'source': 'basic-text.pdf', 'page_label': '1', 'title': 'Sample Document for PDF Testing', 'producer': 'Skia/PDF m126'}
TEXT: Sample Document for PDF Testing
Introduction
This is a simple document created to test basic PDF functionality. It includes various text formatting
options to ensure proper rendering in PDF readers.
Text Formatting Examples
1. Bold text is used for emphasis.
2. Italic text can be used for titles or subtle emph

In [28]:
query = "What is example of unordered list?"

query_embedding = embedding_model.encode(
    [query]
)

results = collection.query(
    query_embeddings=query_embedding.tolist(),
    n_results=3
)

In [29]:
for i in range(len(results["documents"][0])):
    print("=" * 80)
    print("Result:", i + 1)
    print("Metadata:", results["metadatas"][0][i])
    print("Text:", results["documents"][0][i][:500])

Result: 1
Metadata: {'source': 'long-doc.txt', 'chunk_id': 79}
Text: elit. Lorem ipsum dolor sit amet, consectetur adipiscing elit. Lorem ipsum dolor sit amet, consectetur adipiscing elit. Lorem ipsum dolor sit amet, consectetur adipiscing elit. Lorem ipsum dolor sit amet, consectetur adipiscing elit. Lorem ipsum dolor sit amet, consectetur adipiscing elit. Lorem ipsum dolor sit amet, consectetur adipiscing elit. Lorem ipsum dolor sit amet, consectetur adipiscing elit. Lorem ipsum dolor sit amet, consectetur adipiscing elit. Lorem ipsum dolor sit amet, consectetu
Result: 2
Metadata: {'chunk_id': 104, 'source': 'long-doc.txt'}
Text: elit. Lorem ipsum dolor sit amet, consectetur adipiscing elit. Lorem ipsum dolor sit amet, consectetur adipiscing elit. Lorem ipsum dolor sit amet, consectetur adipiscing elit. Lorem ipsum dolor sit amet, consectetur adipiscing elit. Lorem ipsum dolor sit amet, consectetur adipiscing elit. Lorem ipsum dolor sit amet, consectetur adipiscing elit. Lorem ipsum 

In [30]:
stage_times = {}

In [31]:
start = time.perf_counter()

documents = load_documents(DOCUMENTS_DIR)

stage_times["load"] = time.perf_counter() - start

In [32]:
start = time.perf_counter()

chunks = text_splitter.split_documents(documents)

stage_times["chunk"] = time.perf_counter() - start

In [33]:
start = time.perf_counter()

texts = [chunk.page_content for chunk in chunks]

embeddings = embedding_model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True
)

stage_times["embed"] = time.perf_counter() - start

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

In [34]:
start = time.perf_counter()

collection.add(
    ids=ids,
    documents=documents_text,
    embeddings=embeddings.tolist(),
    metadatas=metadatas
)

stage_times["store"] = time.perf_counter() - start

In [35]:
print("\nStage Timing")
print("-" * 40)

for stage, duration in stage_times.items():
    print(f"{stage:<10}: {duration:.3f} seconds")


Stage Timing
----------------------------------------
load      : 0.433 seconds
chunk     : 0.039 seconds
embed     : 12.945 seconds
store     : 0.071 seconds
